# 🔬 Research Insights Platform

### Part 1 — Backend Foundation (Semantic Search Engine)

**Goal of this project (overall):** Build a platform that analyzes research papers and generates useful *insights* — semantic search is just one small piece of it. Later parts will add trend analysis, keyword extraction, NER, and visualizations.

**Goal of Part 1 (this notebook):** Build only the foundation:
1. Load and clean the dataset
2. Generate sentence embeddings for each paper
3. Build a FAISS vector index
4. Create a working semantic search function

No trend analysis, no keyword extraction, no NER, no visualizations, no dashboards yet. Just a clean, working search backend that later parts will build on top of.

---

## 1. Install Required Libraries

We need:
- `datasets` → to load the HuggingFace dataset
- `sentence-transformers` → to turn text into embeddings (numeric vectors)
- `faiss-cpu` → to do fast similarity search over those embeddings
- `pandas` → to work with tabular data

Colab already has `pandas` and `numpy`, but we install the rest explicitly.

In [5]:
# Install libraries not pre-installed in Colab
# -q keeps the output quiet/clean
!pip install -q datasets sentence-transformers faiss-cpu

In [6]:
# Import all libraries used in this notebook
import re
import numpy as np
import pandas as pd

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss

# Nicely display long text in pandas without cutting it off
pd.set_option('display.max_colwidth', 200)

## 2. Load the Dataset

We use the `CShorten/ML-ArXiv-Papers` dataset from HuggingFace, which contains machine learning research papers with their `title` and `abstract`.

To keep things fast for a portfolio project, we limit ourselves to roughly **15,000 papers**.

In [7]:
# Number of papers we want to work with (kept small for speed in Colab)
NUM_PAPERS = 15000

# Load the dataset directly from HuggingFace
# The dataset has a single 'train' split
raw_dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")

print(f"Full dataset size: {len(raw_dataset)} papers")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Full dataset size: 117592 papers


## 3. Convert Dataset into a Pandas DataFrame

Pandas DataFrames are much easier to filter, clean, and inspect than a raw HuggingFace `Dataset` object.

In [8]:
# Convert to pandas DataFrame
df = raw_dataset.to_pandas()

# Keep only the first NUM_PAPERS rows for faster processing
df = df.head(NUM_PAPERS).reset_index(drop=True)

print(f"Working with {len(df)} papers")
print(f"Columns available: {list(df.columns)}")
df.head(3)

Working with 15000 papers
Columns available: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract']


,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint...
1,1,1.0,Sensor Networks with Random Links: Topology Design for Distributed\n Consensus,"In a sensor network, in practice, the communication among sensors is subject\nto:(1) errors or failures at random times; (3) costs; and(2) constraints since\nsensors and networks operate under s..."
2,2,2.0,The on-line shortest path problem under partial monitoring,The on-line shortest path problem is considered under various models of\npartial monitoring. Given a weighted directed acyclic graph whose edge weights\ncan change in an arbitrary (adversarial) ...


## 4. Keep Only `title` and `abstract`

The raw dataset may contain extra columns we don't need. We only care about the paper's `title` and `abstract`, since those are what we'll search over and analyze.

In [9]:
# Some versions of this dataset name the abstract column 'abstract' and
# sometimes there's an extra unnamed index column. We handle that safely here.

# Standardize column names to lowercase for safety
df.columns = [c.strip().lower() for c in df.columns]

# Keep only the two columns we actually need
df = df[["title", "abstract"]].copy()

# Drop rows where title or abstract is missing
df = df.dropna(subset=["title", "abstract"]).reset_index(drop=True)

print(f"Papers remaining after dropping missing values: {len(df)}")
df.head(3)

Papers remaining after dropping missing values: 15000


,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to construct a predictor of a random\nvariable $Y$ as a function of a related random variable $X$ on the basis of an\ni.i.d. training sample from the joint...
1,Sensor Networks with Random Links: Topology Design for Distributed\n Consensus,"In a sensor network, in practice, the communication among sensors is subject\nto:(1) errors or failures at random times; (3) costs; and(2) constraints since\nsensors and networks operate under s..."
2,The on-line shortest path problem under partial monitoring,The on-line shortest path problem is considered under various models of\npartial monitoring. Given a weighted directed acyclic graph whose edge weights\ncan change in an arbitrary (adversarial) ...


## 5. Clean the Text

Raw text from arXiv abstracts often contains:
- Extra whitespace / newline characters
- LaTeX-style symbols
- Odd punctuation spacing

We write a simple, reusable cleaning function and apply it to both `title` and `abstract`.

In [10]:
def clean_text(text: str) -> str:
    """
    Clean a piece of text by:
    - Removing newline / tab characters
    - Collapsing multiple spaces into one
    - Stripping leading/trailing whitespace

    This is intentionally simple and beginner-friendly.
    """
    if not isinstance(text, str):
        return ""

    # Replace newlines and tabs with a single space
    text = re.sub(r"[\n\t\r]+", " ", text)

    # Collapse multiple spaces into a single space
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text


# Apply the cleaning function to both columns
df["title"] = df["title"].apply(clean_text)
df["abstract"] = df["abstract"].apply(clean_text)

df.head(3)

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint dis...
1,Sensor Networks with Random Links: Topology Design for Distributed Consensus,"In a sensor network, in practice, the communication among sensors is subject to:(1) errors or failures at random times; (3) costs; and(2) constraints since sensors and networks operate under scarc..."
2,The on-line shortest path problem under partial monitoring,"The on-line shortest path problem is considered under various models of partial monitoring. Given a weighted directed acyclic graph whose edge weights can change in an arbitrary (adversarial) way,..."


## 6. Create the `paper_text` Column

For semantic search, we want to embed a combination of the title and abstract, since the title often adds useful context that the abstract alone might not fully capture.

In [11]:
# Combine title and abstract into a single text field used for embeddings
df["paper_text"] = df["title"] + ". " + df["abstract"]

# Quick sanity check
print(df["paper_text"].iloc[0][:300], "...")

Learning from compressed observations. The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some sp ...


## 7. Load the SentenceTransformer Model

We use `all-MiniLM-L6-v2` — a small, fast, and popular sentence embedding model that works well for semantic search and is a great fit for Colab (light on memory, quick to run).

In [12]:
# Load a lightweight, well-known sentence embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded successfully.")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

Model loaded successfully.
Embedding dimension: 384


/tmp/ipykernel_2548/1626100498.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")


## 8. Generate Embeddings

We convert every paper's `paper_text` into a numeric vector (embedding). This is what allows us to compare papers by *meaning* rather than exact keyword matches.

This step can take a few minutes depending on Colab's hardware — using a GPU runtime will make it much faster.

In [13]:
import os

embeddings_file = 'paper_embeddings.npy'

if os.path.exists(embeddings_file):
    print(f'Loading embeddings from {embeddings_file}...')
    paper_embeddings = np.load(embeddings_file)
    print(f'Loaded embeddings shape: {paper_embeddings.shape}')
else:
    print('Generating new embeddings...')
    # Generate embeddings for all papers
    # show_progress_bar=True gives visual feedback since this can take a while
    # normalize_embeddings=True makes cosine similarity search easier and more accurate
    paper_embeddings = embedding_model.encode(
        df["paper_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    # Convert to float32 as FAISS requires it
    paper_embeddings = np.array(paper_embeddings).astype("float32")

    print(f"Generated embeddings shape: {paper_embeddings.shape}")
    np.save(embeddings_file, paper_embeddings)
    print(f'Embeddings saved to {embeddings_file}')

Loading embeddings from paper_embeddings.npy...
Loaded embeddings shape: (15000, 384)


## 9. Save Embeddings in Memory

We keep the embeddings as a `float32` NumPy array in memory (this is what FAISS expects). In later parts, we could save these to disk so we don't need to regenerate them every time — but for Part 1, keeping them in memory is enough.

In [14]:
# FAISS requires float32 NumPy arrays
paper_embeddings = np.array(paper_embeddings).astype("float32")

print(f"Embeddings stored in memory. Shape: {paper_embeddings.shape}, dtype: {paper_embeddings.dtype}")

Embeddings stored in memory. Shape: (15000, 384), dtype: float32


In [15]:
np.save('paper_embeddings.npy', paper_embeddings)
print('Embeddings saved to paper_embeddings.npy')

Embeddings saved to paper_embeddings.npy


To load the embeddings back in a new session, you can use:

```python
import numpy as np
paper_embeddings = np.load('paper_embeddings.npy')
```

## 10. Build the FAISS Index

FAISS (Facebook AI Similarity Search) lets us search millions of vectors very quickly.

Since we normalized our embeddings in Step 8, we use `IndexFlatIP` (Inner Product). For normalized vectors, inner product is equivalent to cosine similarity — a common and simple approach for semantic search.

In [16]:
# Get the dimensionality of our embeddings (e.g. 384 for MiniLM)
embedding_dim = paper_embeddings.shape[1]

# Create a FAISS index that uses inner product similarity
# (equivalent to cosine similarity, since our vectors are normalized)
faiss_index = faiss.IndexFlatIP(embedding_dim)

# Add all our paper embeddings to the index
faiss_index.add(paper_embeddings)

print(f"FAISS index built successfully. Total vectors indexed: {faiss_index.ntotal}")

FAISS index built successfully. Total vectors indexed: 15000


In [17]:
faiss.write_index(faiss_index, 'faiss_index.bin')
print('FAISS index saved to faiss_index.bin')

FAISS index saved to faiss_index.bin


To load the FAISS index back in a new session, you can use:

```python
import faiss
faiss_index = faiss.read_index('faiss_index.bin')
```

## 11–15. Semantic Search Function

Now we tie everything together into one reusable function, `semantic_search()`, which:
1. Takes a user's text query
2. Converts it into an embedding (same model, same normalization)
3. Searches the FAISS index for the most similar papers
4. Returns the Top-K most relevant papers, along with a similarity score

In [18]:
def semantic_search(query: str, top_k: int = 5) -> pd.DataFrame:
    """
    Search for the most semantically similar papers to a given query.

    Steps:
    1. Convert the query into an embedding using the same model used for papers.
    2. Search the FAISS index for the top_k closest paper embeddings.
    3. Build a results DataFrame with title, abstract, and similarity score.

    Args:
        query (str): The user's search query, e.g. "transformers for text classification"
        top_k (int): Number of top results to return

    Returns:
        pd.DataFrame: Columns = ['title', 'abstract', 'similarity_score']
    """
    # Step 1: Clean and embed the query the same way papers were embedded
    clean_query = clean_text(query)
    query_embedding = embedding_model.encode(
        [clean_query],
        normalize_embeddings=True,
    ).astype("float32")

    # Step 2: Search FAISS index for the top_k most similar papers
    # 'scores' = similarity scores, 'indices' = row positions in our DataFrame
    scores, indices = faiss_index.search(query_embedding, top_k)

    # Step 3: Collect results into a clean DataFrame
    results = df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]

    # Reset index for clean display
    results = results.reset_index(drop=True)

    return results[["title", "abstract", "similarity_score"]]

## 16. Try It Out — Enter a Query and Print Results

Let's test the search function with an example query. Feel free to change `user_query` and re-run this cell.

In [19]:
def print_search_results(query: str, top_k: int = 5):
    """
    Run a semantic search and neatly print the results:
    Title, Similarity Score, and Abstract for each result.
    """
    results = semantic_search(query, top_k=top_k)

    print(f"Search Query: \"{query}\"\n")
    print("=" * 100)

    for i, row in results.iterrows():
        print(f"\nResult {i + 1}")
        print(f"Title: {row['title']}")
        print(f"Similarity Score: {row['similarity_score']:.4f}")
        print(f"Abstract: {row['abstract']}")
        print("-" * 100)


# Example usage — try changing this query!
user_query = "transformer models for text classification"
print_search_results(user_query, top_k=5)

Search Query: "transformer models for text classification"


Result 1
Title: Naive Bayes and Text Classification I - Introduction and Theory
Similarity Score: 0.5948
Abstract: Naive Bayes classifiers, a family of classifiers that are based on the popular Bayes' probability theorem, are known for creating simple yet well performing models, especially in the fields of document classification and disease prediction. In this article, we will look at the main concepts of naive Bayes classification in the context of document categorization.
----------------------------------------------------------------------------------------------------

Result 2
Title: Supervised learning Methods for Bangla Web Document Categorization
Similarity Score: 0.5534
Abstract: This paper explores the use of machine learning approaches, or more specifically, four supervised learning Methods, namely Decision Tree(C 4.5), K-Nearest Neighbour (KNN), Na\"ive Bays (NB), and Support Vector Machine (SVM) for categorizat

---
## ✅ Part 1 Complete

We now have a working backend foundation:
- Cleaned dataset (`df`) with `title`, `abstract`, `paper_text`
- Sentence embeddings (`paper_embeddings`)
- A FAISS index (`faiss_index`)
- A reusable `semantic_search()` function

This is only the foundation. Part 2 (not built yet) will add analysis features like trend analysis, keyword extraction, and NER on top of this same dataset and embeddings.

In [20]:
!pip uninstall -y transformers
!pip install transformers==4.55.4 accelerate sentencepiece

Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
  Successfully uninstalled transformers-4.55.4
  Using cached transformers-4.55.4-py3-none-any.whl.metadata (41 kB)
Using cached transformers-4.55.4-py3-none-any.whl (11.3 MB)


In [21]:
!pip install keybert
from transformers import pipeline
from keybert import KeyBERT
import spacy
from collections import Counter

summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
kw_model = KeyBERT(model=embedding_model)
nlp = spacy.load("en_core_web_sm")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [22]:
def summarize_paper(text, max_length=60, min_length=15):
    text = clean_text(text)
    if len(text.split()) < 20:
        return text
    input_text = " ".join(text.split()[:400])
    summary = summarizer(input_text, max_length=max_length, min_length=min_length, do_sample=False)
    return summary[0]["summary_text"]

In [23]:
def extract_keywords(text, top_n=5):
    text = clean_text(text)
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_n,
    )
    return [kw for kw, score in keywords]

In [24]:
MODEL_TERMS = {"bert", "gpt", "transformer", "lstm", "cnn", "rnn", "resnet", "gan",
               "bert-base", "roberta", "t5", "vae", "vgg", "yolo", "xlnet", "alexnet",
               "unet", "bart", "albert", "distilbert"}

TECH_TERMS = {"deep learning", "machine learning", "neural network", "reinforcement learning",
              "computer vision", "nlp", "natural language processing", "attention mechanism",
              "gradient descent", "supervised learning", "unsupervised learning",
              "transfer learning", "federated learning", "self-supervised learning"}

DATASET_TERMS = {"imagenet", "mnist", "cifar", "coco", "squad", "glue", "wikitext",
                  "cifar-10", "cifar-100", "librispeech", "penn treebank"}

def extract_entities(text):
    text = clean_text(text)
    doc = nlp(text)
    text_lower = text.lower()

    entities = {
        "models": set(),
        "organizations": set(),
        "technologies": set(),
        "datasets": set(),
        "researchers": set(),
    }

    for ent in doc.ents:
        ent_text = ent.text.strip()
        if not ent_text:
            continue
        if ent.label_ == "ORG":
            entities["organizations"].add(ent_text)
        elif ent.label_ == "PERSON":
            entities["researchers"].add(ent_text)

    for term in MODEL_TERMS:
        if term in text_lower:
            entities["models"].add(term)

    for term in TECH_TERMS:
        if term in text_lower:
            entities["technologies"].add(term)

    for term in DATASET_TERMS:
        if term in text_lower:
            entities["datasets"].add(term)

    return {k: sorted(v) for k, v in entities.items()}

In [25]:
def display_enriched_results(query, top_k=5):
    results = semantic_search(query, top_k=top_k)

    print(f"Search Query: \"{query}\"\n")
    print("=" * 100)

    for i, row in results.iterrows():
        print(f"\nResult {i + 1}")
        print(f"Title: {row['title']}")
        print(f"Similarity Score: {row['similarity_score']:.4f}")
        print(f"Abstract: {row['abstract']}")

        summary = summarize_paper(row["abstract"])
        print(f"\nSummary: {summary}")

        keywords = extract_keywords(row["abstract"])
        print(f"Top Keywords: {', '.join(keywords)}")

        entities = extract_entities(row["abstract"])
        print("Entities:")
        for ent_type, ent_list in entities.items():
            if ent_list:
                print(f"  {ent_type.capitalize()}: {', '.join(ent_list)}")

        print("-" * 100)

    return results


user_query = "transformer models for text classification"
enriched_results = display_enriched_results(user_query, top_k=5)

Search Query: "transformer models for text classification"


Result 1
Title: Naive Bayes and Text Classification I - Introduction and Theory
Similarity Score: 0.5948
Abstract: Naive Bayes classifiers, a family of classifiers that are based on the popular Bayes' probability theorem, are known for creating simple yet well performing models, especially in the fields of document classification and disease prediction. In this article, we will look at the main concepts of naive Bayes classification in the context of document categorization.

Summary:  Naive Bayes classifiers are based on the popular Bayes' probability theorem . They are known for creating simple yet well performing models, especially in the fields of document classification and disease prediction . In this article, we will look at the main concepts of naive Bayes classification in the
Top Keywords: bayes classifiers, bayes classification, naive bayes, document classification, document categorization
Entities:
  Organizations

In [26]:
def generate_insights(results):
    all_keywords = []
    all_entities = {"models": [], "organizations": [], "technologies": [], "datasets": [], "researchers": []}

    for _, row in results.iterrows():
        all_keywords.extend(extract_keywords(row["abstract"]))
        entities = extract_entities(row["abstract"])
        for ent_type in all_entities:
            all_entities[ent_type].extend(entities[ent_type])

    keyword_counts = Counter(all_keywords)
    entity_counts = {k: Counter(v) for k, v in all_entities.items()}

    print("Research Insights")
    print("=" * 100)

    print("\nMost Common Keywords:")
    for kw, count in keyword_counts.most_common(5):
        print(f"  {kw}: {count}")

    print("\nMost Discussed Technologies:")
    for ent, count in entity_counts["technologies"].most_common(5):
        print(f"  {ent}: {count}")

    print("\nMost Discussed Organizations:")
    for ent, count in entity_counts["organizations"].most_common(5):
        print(f"  {ent}: {count}")

    print("\nMost Discussed Datasets:")
    for ent, count in entity_counts["datasets"].most_common(5):
        print(f"  {ent}: {count}")

    print("\nMost Discussed Models:")
    for ent, count in entity_counts["models"].most_common(5):
        print(f"  {ent}: {count}")

    return {
        "keyword_counts": keyword_counts,
        "entity_counts": entity_counts,
    }


insights = generate_insights(enriched_results)

Research Insights

Most Common Keywords:
  document classification: 2
  document categorization: 2
  text classification: 2
  bayes classifiers: 1
  bayes classification: 1

Most Discussed Technologies:
  machine learning: 2
  supervised learning: 2
  gradient descent: 1

Most Discussed Organizations:
  SVM: 2
  Bayes: 1
  Naive Bayes: 1
  Bays (NB: 1
  K-Nearest Neighbour: 1

Most Discussed Datasets:

Most Discussed Models:


In [27]:
def compare_topics(topic1, topic2, top_k=5):
    results1 = semantic_search(topic1, top_k=top_k)
    results2 = semantic_search(topic2, top_k=top_k)

    def topic_summary(results):
        all_keywords = []
        all_entities = []
        for _, row in results.iterrows():
            all_keywords.extend(extract_keywords(row["abstract"]))
            entities = extract_entities(row["abstract"])
            for ent_list in entities.values():
                all_entities.extend(ent_list)
        return {
            "top_keywords": [kw for kw, _ in Counter(all_keywords).most_common(5)],
            "top_entities": [ent for ent, _ in Counter(all_entities).most_common(5)],
            "avg_similarity": results["similarity_score"].mean(),
        }

    summary1 = topic_summary(results1)
    summary2 = topic_summary(results2)

    print(f"Comparison: \"{topic1}\" vs \"{topic2}\"")
    print("=" * 100)

    print(f"\nTop Keywords ({topic1}): {', '.join(summary1['top_keywords'])}")
    print(f"Top Keywords ({topic2}): {', '.join(summary2['top_keywords'])}")

    print(f"\nTop Entities ({topic1}): {', '.join(summary1['top_entities'])}")
    print(f"Top Entities ({topic2}): {', '.join(summary2['top_entities'])}")

    print(f"\nAverage Similarity Score ({topic1}): {summary1['avg_similarity']:.4f}")
    print(f"Average Similarity Score ({topic2}): {summary2['avg_similarity']:.4f}")

    more_relevant = topic1 if summary1["avg_similarity"] > summary2["avg_similarity"] else topic2
    print(f"\nSummary: \"{more_relevant}\" has stronger and more consistent matches in this dataset. "
          f"\"{topic1}\" and \"{topic2}\" show {'overlapping' if set(summary1['top_keywords']) & set(summary2['top_keywords']) else 'distinct'} keyword themes.")

    return summary1, summary2


compare_topics("transformer models", "reinforcement learning")

Comparison: "transformer models" vs "reinforcement learning"

Top Keywords (transformer models): transformer networks, deep learning, learning shapes, dense prediction, dense transformer
Top Keywords (reinforcement learning): reinforcement learning, optimal policy, reward optimal, markovian knowing, mdp

Top Entities (transformer models): transformer, deep learning, machine learning, Inverse Compositional Spatial Transformer Networks, LK
Top Entities (reinforcement learning): reinforcement learning, RL, MDP, Markovian, the Reinforcement Learning

Average Similarity Score (transformer models): 0.3383
Average Similarity Score (reinforcement learning): 0.5849

Summary: "reinforcement learning" has stronger and more consistent matches in this dataset. "transformer models" and "reinforcement learning" show distinct keyword themes.


({'top_keywords': ['transformer networks',
   'deep learning',
   'learning shapes',
   'dense prediction',
   'dense transformer'],
  'top_entities': ['transformer',
   'deep learning',
   'machine learning',
   'Inverse Compositional Spatial Transformer Networks',
   'LK'],
  'avg_similarity': np.float32(0.3383295)},
 {'top_keywords': ['reinforcement learning',
   'optimal policy',
   'reward optimal',
   'markovian knowing',
   'mdp'],
  'top_entities': ['reinforcement learning',
   'RL',
   'MDP',
   'Markovian',
   'the Reinforcement Learning'],
  'avg_similarity': np.float32(0.5848873)})

In [28]:
def research_statistics(results):
    abstract_lengths = results["abstract"].apply(lambda x: len(x.split()))
    all_keywords = []
    all_entities = []

    for _, row in results.iterrows():
        all_keywords.extend(extract_keywords(row["abstract"]))
        entities = extract_entities(row["abstract"])
        for ent_list in entities.values():
            all_entities.extend(ent_list)

    keyword_counts = Counter(all_keywords)
    entity_counts = Counter(all_entities)

    most_frequent_keyword = keyword_counts.most_common(1)[0][0] if keyword_counts else "N/A"
    most_frequent_entity = entity_counts.most_common(1)[0][0] if entity_counts else "N/A"

    print("Research Statistics")
    print("=" * 100)
    print(f"Total Papers Searched: {len(results)}")
    print(f"Average Abstract Length (words): {abstract_lengths.mean():.1f}")
    print(f"Average Similarity Score: {results['similarity_score'].mean():.4f}")
    print(f"Most Frequent Keyword: {most_frequent_keyword}")
    print(f"Most Frequent Entity: {most_frequent_entity}")

    return {
        "total_papers": len(results),
        "avg_abstract_length": abstract_lengths.mean(),
        "avg_similarity_score": results["similarity_score"].mean(),
        "most_frequent_keyword": most_frequent_keyword,
        "most_frequent_entity": most_frequent_entity,
    }


research_statistics(enriched_results)

Research Statistics
Total Papers Searched: 5
Average Abstract Length (words): 138.4
Average Similarity Score: 0.5553
Most Frequent Keyword: document classification
Most Frequent Entity: SVM


{'total_papers': 5,
 'avg_abstract_length': np.float64(138.4),
 'avg_similarity_score': np.float32(0.5553217),
 'most_frequent_keyword': 'document classification',
 'most_frequent_entity': 'SVM'}

In [29]:
import re as _re

def _extract_topics_from_text(text, n=2):
    stop_phrases = [
        "compare", "vs", "versus", "and", "papers", "paper", "about",
        "summarize", "summarise", "search", "find", "show", "give",
        "keywords", "keyword", "entities", "entity", "organizations",
        "organisation", "insights", "insight", "statistics", "stats",
        "trends", "trend", "current", "what", "are", "the", "is", "for",
        "on", "in", "of", "me", "common", "mentioned", "extract",
    ]
    cleaned = _re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower())
    words = [w for w in cleaned.split() if w not in stop_phrases]
    topic_text = " ".join(words).strip()
    return topic_text if topic_text else text

In [30]:
def research_agent(user_request, top_k=5):
    """
    Lightweight intent-routing agent.
    Reads a natural language request and calls the correct existing
    function(s) from the notebook (semantic_search, summarize_paper,
    extract_keywords, extract_entities, generate_insights, compare_topics,
    research_statistics). No external agent framework is used.
    """
    request_lower = user_request.lower()

    print(f"Agent received request: \"{user_request}\"")
    print("=" * 100)

    # 1. Topic comparison
    compare_match = _re.search(
        r"compare\s+(.+?)\s+(?:and|vs|versus)\s+(.+)", request_lower
    )
    if "compare" in request_lower or " vs " in request_lower or "versus" in request_lower:
        if compare_match:
            topic1 = compare_match.group(1).strip(" ?.!")
            topic2 = compare_match.group(2).strip(" ?.!")
            print(f"Agent decision: Compare Topics Tool -> ({topic1}) vs ({topic2})\n")
            return compare_topics(topic1, topic2, top_k=top_k)
        else:
            print("Agent could not detect two topics to compare. Please phrase as 'Compare X and Y'.")
            return None

    # 2. Research statistics
    if "statistic" in request_lower or "stats" in request_lower:
        topic = _extract_topics_from_text(user_request)
        print(f"Agent decision: Semantic Search -> Research Statistics Tool (topic: {topic})\n")
        results = semantic_search(topic, top_k=top_k)
        return research_statistics(results)

    # 3. Named entity related requests (organizations, entities, models, datasets, researchers)
    entity_signal_words = ["entit", "organization", "organisation", "researcher", "dataset", "model mentioned"]
    if any(word in request_lower for word in entity_signal_words):
        topic = _extract_topics_from_text(user_request)
        print(f"Agent decision: Semantic Search -> NER Tool (topic: {topic})\n")
        results = semantic_search(topic, top_k=top_k)
        all_entities = {"models": [], "organizations": [], "technologies": [], "datasets": [], "researchers": []}
        for _, row in results.iterrows():
            entities = extract_entities(row["abstract"])
            for ent_type in all_entities:
                all_entities[ent_type].extend(entities[ent_type])
        for ent_type, ent_list in all_entities.items():
            unique_entities = sorted(set(ent_list))
            if unique_entities:
                print(f"{ent_type.capitalize()}: {', '.join(unique_entities)}")
        return all_entities

    # 4. Trend / insight requests (search + keywords + NER + insight generation)
    if "trend" in request_lower or "insight" in request_lower:
        topic = _extract_topics_from_text(user_request)
        print(f"Agent decision: Semantic Search -> Keyword Extraction -> NER -> Insight Generation (topic: {topic})\n")
        results = semantic_search(topic, top_k=top_k)
        return generate_insights(results)

    # 5. Keyword extraction requests
    if "keyword" in request_lower:
        topic = _extract_topics_from_text(user_request)
        print(f"Agent decision: Semantic Search -> Keyword Extraction Tool (topic: {topic})\n")
        results = semantic_search(topic, top_k=top_k)
        for _, row in results.iterrows():
            keywords = extract_keywords(row["abstract"])
            print(f"Title: {row['title']}")
            print(f"Keywords: {', '.join(keywords)}")
            print("-" * 100)
        return results

    # 6. Summarization requests
    if "summar" in request_lower:
        topic = _extract_topics_from_text(user_request)
        print(f"Agent decision: Semantic Search -> Summarization Tool (topic: {topic})\n")
        results = semantic_search(topic, top_k=top_k)
        for _, row in results.iterrows():
            summary = summarize_paper(row["abstract"])
            print(f"Title: {row['title']}")
            print(f"Summary: {summary}")
            print("-" * 100)
        return results

    # 7. Default: plain semantic search
    topic = _extract_topics_from_text(user_request)
    print(f"Agent decision: Semantic Search Tool (topic: {topic})\n")
    return print_search_results(topic, top_k=top_k)

In [31]:
research_agent("Compare Transformers and Diffusion Models")

Agent received request: "Compare Transformers and Diffusion Models"
Agent decision: Compare Topics Tool -> (transformers) vs (diffusion models)

Comparison: "transformers" vs "diffusion models"

Top Keywords (transformers): transformer networks, learning shapes, dense prediction, deep learning, dense transformer
Top Keywords (diffusion models): diffusion network, diffusion content, nodes diffusion, predicting diffusion, temporal diffusion

Top Entities (transformers): transformer, deep learning, Fourier, Hadamard etc., Givens, neural network
Top Entities (diffusion models): Markov, Pouget-Abadie, the Independent Cascade, p)$, KLMS

Average Similarity Score (transformers): 0.3311
Average Similarity Score (diffusion models): 0.4754

Summary: "diffusion models" has stronger and more consistent matches in this dataset. "transformers" and "diffusion models" show distinct keyword themes.


({'top_keywords': ['transformer networks',
   'learning shapes',
   'dense prediction',
   'deep learning',
   'dense transformer'],
  'top_entities': ['transformer',
   'deep learning',
   'Fourier, Hadamard etc.',
   'Givens',
   'neural network'],
  'avg_similarity': np.float32(0.3310689)},
 {'top_keywords': ['diffusion network',
   'diffusion content',
   'nodes diffusion',
   'predicting diffusion',
   'temporal diffusion'],
  'top_entities': ['Markov',
   'Pouget-Abadie',
   'the Independent Cascade',
   'p)$',
   'KLMS'],
  'avg_similarity': np.float32(0.47538215)})

In [32]:
research_agent("Summarize papers about Reinforcement Learning")

Agent received request: "Summarize papers about Reinforcement Learning"
Agent decision: Semantic Search -> Summarization Tool (topic: reinforcement learning)

Title: Selecting the State-Representation in Reinforcement Learning
Summary:  The problem of selecting the right state-representation in a reinforcement learning problem is considered . Several models (functions mapping past observations to a finite set) of the observations are given . It is known that for at least one of these models the resulting state dynamics are Markovian .
----------------------------------------------------------------------------------------------------
Title: PAC Reinforcement Learning with Rich Observations
Summary:  We propose and study a new model for reinforcement learning with rich observations . These models require an agent to take actions based on observations (features) with the goal of achieving long-term performance competitive with a large set of policies . We prove that the algorithm learns 

,title,abstract,similarity_score
0,Selecting the State-Representation in Reinforcement Learning,The problem of selecting the right state-representation in a reinforcement learning problem is considered. Several models (functions mapping past observations to a finite set) of the observations ...,0.593867
1,PAC Reinforcement Learning with Rich Observations,"We propose and study a new model for reinforcement learning with rich observations, generalizing contextual bandits to sequential decision making. These models require an agent to take actions bas...",0.586019
2,Reinforcement Learning algorithms for regret minimization in structured Markov Decision Processes,A recent goal in the Reinforcement Learning (RL) framework is to choose a sequence of actions or a policy to maximize the reward collected or minimize the regret incurred in a finite time horizon....,0.582996
3,Reinforcement Learning via AIXI Approximation,"This paper introduces a principled approach for the design of a scalable general reinforcement learning agent. This approach is based on a direct approximation of AIXI, a Bayesian optimality notio...",0.582804
4,Angrier Birds: Bayesian reinforcement learning,We train a reinforcement learner to play a simplified version of the game Angry Birds. The learner is provided with a game state in a manner similar to the output that could be produced by compute...,0.578750


In [33]:
research_agent("What are the common organizations mentioned in GPT research?")

Agent received request: "What are the common organizations mentioned in GPT research?"
Agent decision: Semantic Search -> NER Tool (topic: gpt research)

Organizations: Bishop, GP, GPLC, GPU, MCTS, SVM, Thompson
Researchers: GPLC, Gaussian, Gaussian Process, Goldberg, Markov Chain Monte Carlo, R. Altogether, Rgtsvm, Williams


{'models': [],
 'organizations': ['GP',
  'GPU',
  'GPU',
  'SVM',
  'GPU',
  'MCTS',
  'Thompson',
  'Bishop',
  'GP',
  'GPLC'],
 'technologies': [],
 'datasets': [],
 'researchers': ['R. Altogether',
  'Rgtsvm',
  'GPLC',
  'Gaussian',
  'Gaussian Process',
  'Goldberg',
  'Markov Chain Monte Carlo',
  'Williams']}

In [34]:
research_agent("What are the current trends in LLM research?")

Agent received request: "What are the current trends in LLM research?"
Agent decision: Semantic Search -> Keyword Extraction -> NER -> Insight Generation (topic: llm research)

Research Insights

Most Common Keywords:
  corporate researchers: 1
  researchers field: 1
  learning researchers: 1
  corporate: 1
  researchers: 1

Most Discussed Technologies:
  machine learning: 4
  gradient descent: 1
  neural network: 1

Most Discussed Organizations:
  PLM: 1
  PSGD: 1
  Perpetual Stochastic Gradient Descent: 1
  Sparse: 1
  linear: 1

Most Discussed Datasets:

Most Discussed Models:


{'keyword_counts': Counter({'corporate researchers': 1,
          'researchers field': 1,
          'learning researchers': 1,
          'corporate': 1,
          'researchers': 1,
          'perpetual learning': 1,
          'selective forgetting': 1,
          'selective memory': 1,
          'perpetual stochastic': 1,
          'driven memory': 1,
          'sparsity paradigm': 1,
          'sparsity sparse': 1,
          'sparsity computational': 1,
          'learning sparsity': 1,
          'sparsity driven': 1,
          'ml research': 1,
          'learning ml': 1,
          'machine learning': 1,
          'impact ml': 1,
          'ml matters': 1,
          'meta database': 1,
          'databases': 1,
          'database provides': 1,
          'database': 1,
          'sql databases': 1}),
 'entity_counts': {'models': Counter(),
  'organizations': Counter({'PLM': 1,
           'PSGD': 1,
           'Perpetual Stochastic Gradient Descent': 1,
           'Sparse': 1,
        

In [38]:
import sys

# LangChain Agent Extension
try:
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage
    from langchain.agents import create_agent
    from langchain_groq import ChatGroq
except ImportError:
    !{sys.executable} -m pip install -qU langchain langchain-core langchain-groq
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage
    from langchain.agents import create_agent
    from langchain_groq import ChatGroq

import os

try:
    from google.colab import userdata
except Exception:
    userdata = None

_api_key = os.environ.get("GROQ_API_KEY")
if not _api_key and userdata is not None:
    try:
        _api_key = userdata.get("GROQ_API_KEY")
    except Exception:
        pass

if not _api_key:
    raise ValueError("GROQ_API_KEY not found.")

os.environ["GROQ_API_KEY"] = _api_key

@tool
def semantic_search_tool(query: str, top_k: int = 5) -> str:
    """Search research papers semantically."""
    # Ensure top_k is an integer, as the agent might pass it as a string
    return semantic_search(query, top_k=int(top_k)).to_string()

@tool
def summarize_paper_tool(text: str) -> str:
    """Summarize a paper abstract."""
    return summarize_paper(text)

@tool
def extract_keywords_tool(text: str) -> str:
    """Extract keywords."""
    return ", ".join(extract_keywords(text))

@tool
def extract_entities_tool(text: str) -> str:
    """Extract named entities."""
    return str(extract_entities(text))

@tool
def generate_insights_tool(query: str) -> str:
    """Generate research insights."""
    results = semantic_search(query)
    from io import StringIO
    import contextlib
    buf=StringIO()
    with contextlib.redirect_stdout(buf):
        generate_insights(results)
    return buf.getvalue()

@tool
def compare_topics_tool(topic1: str, topic2: str) -> str:
    """Compare two research topics."""
    from io import StringIO
    import contextlib
    buf=StringIO()
    with contextlib.redirect_stdout(buf):
        compare_topics(topic1, topic2)
    return buf.getvalue()

@tool
def research_statistics_tool(query: str) -> str:
    """Show research statistics."""
    results = semantic_search(query)
    from io import StringIO
    import contextlib
    buf=StringIO()
    with contextlib.redirect_stdout(buf):
        research_statistics(results)
    return buf.getvalue()

llm = ChatGroq(model="llama-3.3-70b-versatile")

agent = create_agent(
    model=llm,
    tools=[
        semantic_search_tool,
        summarize_paper_tool,
        extract_keywords_tool,
        extract_entities_tool,
        generate_insights_tool,
        compare_topics_tool,
        research_statistics_tool,
    ],
)

def ask_agent(query):
    response = agent.invoke({"messages":[HumanMessage(content=query)]})
    try:
        print(response["messages"][-1].content)
    except Exception:
        print(response)


In [39]:

ask_agent("Find papers on Reinforcement Learning")
ask_agent("Summarize papers about Vision Transformers")
ask_agent("Compare Transformers and Diffusion Models")



A range of research papers were found on the topic of Reinforcement Learning, with various approaches to solving the problem of selecting the right state-representation in a reinforcement learning problem. The papers proposed different algorithms and models to achieve optimal policies and maximize rewards. 

Some of the key findings and insights from the research papers include:

1. A new model for reinforcement learning with rich observations, generalizing contextual bandits to sequential decision making.
2. An algorithm for selecting the right state-representation in a reinforcement learning problem, with a regret of order T^{2/3} where T is the horizon time.
3. A reinforcement learning algorithm that achieves near optimal behavior after a number of episodes that is polynomial in all relevant parameters, logarithmic in the number of policies, and independent of the size of the observation space.
4. A principled approach for the design of a scalable general reinforcement learning agen

Your max_length is set to 60, but your input_length is only 30. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=15)


The comparison between Transformers and Diffusion Models reveals that both models have their strengths and weaknesses. Transformers are known for their ability to handle sequential data and have been widely used in natural language processing tasks, while Diffusion Models have been found to be effective in modeling complex distributions and have been used in image and video generation tasks. The research insights suggest that Diffusion Models have a stronger presence in the field, with more papers and researchers working on this topic. The summary of the paper highlights the importance of understanding the differences between these two models and their applications in different fields. The extracted keywords and entities provide further insight into the topics and concepts discussed in the paper. The semantic search results provide a list of relevant papers related to the topic, with their titles, abstracts, and similarity scores. Overall, the results suggest that both Transformers and

BadRequestError: Error code: 400 - {'error': {'message': 'tool call validation failed: parameters for tool semantic_search_tool did not match schema: errors: [`/top_k`: expected integer, but got string]', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=semantic_search_tool>{"query": "Large Language Models", "top_k": "5"}</function>'}}